In [2]:
#579
import pandas as pd
data = [[1, 1, 20], [2, 1, 20], [1, 2, 30], [2, 2, 30], [3, 2, 40], [1, 3,
40], [3, 3, 60], [1, 4, 60], [3, 4, 70], [1, 7, 90], [1, 8, 90]]
df= pd.DataFrame(data, columns=['id'
,
'month'
,
'salary']).astype({'id':'Int64'
,
'month':'Int64'
,
'salary':'Int64'})
def cumulative_salary(df):
    result = []

    for emp_id, group in df.groupby('id'):
        months = group['month']
        min_m, max_m = months.min(), months.max()

        # 补全月份
        all_months = pd.DataFrame({'month': range(min_m, max_m + 1)})
        merged = all_months.merge(group, on='month', how='left').fillna({'salary': 0})
        merged['salary'] = merged['salary'].astype(int)

        # 滚动 3 月工资
        merged['Salary'] = merged['salary'].rolling(3, min_periods=1).sum()

        # 去掉最大月份
        latest = max_m
        merged = merged[merged['month'] != latest]

        # 只保留员工原本有工资的月份
        merged = merged[merged['month'].isin(months)]

        # 若还有数据，则加入 result
        if not merged.empty:
            merged['id'] = emp_id
            result.append(merged[['id','month','Salary']])

    # 如果 result 有内容再 concat
    if result:
        final = pd.concat(result)
        final = final.sort_values(['id', 'month'], ascending=[True, False])
    else:
        final = pd.DataFrame(columns=['id','month','Salary'])

    return final

cumulative_salary(df)

,id,month,Salary
6,1,7,90.0
3,1,4,130.0
2,1,3,90.0
1,1,2,50.0
0,1,1,20.0
0,2,1,20.0
1,3,3,100.0
0,3,2,40.0


In [14]:
data = [[1, 1, 20], [2, 1, 20], [1, 2, 30], [2, 2, 30], [3, 2, 40], [1, 3, 40], [3, 3, 60], [1, 4, 60], [3, 4, 70], [1, 7, 90], [1, 8, 90]]
employee = pd.DataFrame(data, columns=['id', 'month', 'salary']).astype({'id':'Int64', 'month':'Int64', 'salary':'Int64'})

id_months = employee.groupby('id')['month'].agg(['min', 'max'])#.reset_index()
id_months
# full_data = pd.concat([
#     pd.DataFrame({'id': row.id, 'month': range(row['min'], row['max'] + 1)})
#     for _, row in id_months.iterrows()
# ], ignore_index=True).merge(employee, how='left').fillna(0)
# full_data = full_data.sort_values(['id', 'month'])
# full_data['Salary'] = full_data.groupby('id')['salary'].rolling(3, min_periods=1).sum().astype(int).reset_index(0, drop=True)
# max_month_map = id_months.set_index('id')['max'].to_dict()
# result = full_data[
#     (full_data['month'] != full_data['id'].map(max_month_map)) & 
#     (full_data['Salary'] != 0)
# ].sort_values(['id', 'month'], ascending=[True, False])[['id', 'month', 'Salary']]
# result

,min,max
id,,
1,1,8
2,1,2
3,2,4


In [5]:
#580
import pandas as pd
data = [[1,
'Jack'
,
'M'
, 1], [2,
'Jane'
,
'F'
, 1], [3,
'Mark'
,
'M'
, 2]]
student = pd.DataFrame(data, columns=['student_id'
,
'student_name'
,
'gender'
,
'dept_id']).astype({'student_id':'Int64'
,
'student_name':'object'
,
'gender':'object'
,
'dept_id':'Int64'})
data = [[1,
'Engineering'], [2,
'Science'], [3,
'Law']]
department = pd.DataFrame(data, columns=['dept_id'
,
'dept_name']).astype({'dept_id':'Int64'
,
'dept_name':'object'})
def department_student_number(student: pd.DataFrame, department: pd.DataFrame) -> pd.DataFrame:
    # 左连接：保证所有部门都出现
    df = department.merge(student, on='dept_id', how='left')

    # 按部门名统计学生数（student_id 为空的要算 0）
    result = df.groupby('dept_name')['student_id'].count().reset_index(name='student_number')

    # 排序：人数降序，再按部门名升序
    result = result.sort_values(['student_number', 'dept_name'], ascending=[False, True])

    return result
department_student_number(student,department)

,dept_name,student_number
0,Engineering,2
2,Science,1
1,Law,0


In [8]:
#597
import pandas as pd
data = [[1, 2,
'2016/06/01'], [1, 3,
'2016/06/01'], [1, 4,
'2016/06/01'],
[2, 3,
'2016/06/02'], [3, 4,
'2016/06/09']]
friend_request = pd.DataFrame(data, columns=['sender_id'
,
'send_to_id'
,
'request_date']).astype({'sender_id':'Int64'
,
'send_to_id':'Int64'
,
'request_date':'datetime64[ns]'})
data = [[1, 2,
'2016/06/03'], [1, 3,
'2016/06/08'], [2, 3,
'2016/06/08'],
[3, 4,
'2016/06/09'], [3, 4,
'2016/06/10']]
request_accepted = pd.DataFrame(data, columns=['requester_id'
,
'accepter_id'
,
'accept_date']).astype({'requester_id':'Int64'
,
'accepter_id':'Int64'
,
'accept_date':'datetime64[ns]'})
# 假设你的数据已读入 df1, df2
df1 = friend_request
df2 = request_accepted

# 1. 求唯一好友申请数
total_requests = df1[['sender_id', 'send_to_id']].drop_duplicates().shape[0]

# 2. 求唯一通过的申请数
accepted_requests = df2[['requester_id', 'accepter_id']].drop_duplicates().shape[0]

# 3. 计算通过率
accept_rate = 0 if total_requests == 0 else round(accepted_requests / total_requests, 2)

# 4. 输出 DataFrame
result = pd.DataFrame({'accept_rate': [accept_rate]})
result


,accept_rate
0,0.8


In [11]:
#1107
import pandas as pd
data = [[1,
'login'
,
'2019-05-01'], [1,
'homepage'
,
'2019-05-01'], [1,
'logout'
,
'2019-05-01'], [2,
'login'
,
'2019-06-21'], [2,
'logout'
,
'2019-06-21'], [3,
'login'
,
'2019-01-01'], [3,
'jobs'
,
'2019-01-01'], [3,
'logout'
,
'2019-01-01'], [4,
'login'
,
'2019-06-21'], [4,
'groups'
,
'2019-06-21'], [4,
'logout'
,
'2019-06-21'], [5,
'login'
,
'2019-03-01'], [5,
'logout'
,
'2019-03-01'], [5,
'login'
,
'2019-06-21'], [5,
'logout'
,
'2019-06-21']]
traffic = pd.DataFrame(data, columns=['user_id'
,
'activity'
,
'activity_date']).astype({'user_id':'Int64'
,
'activity':'object'
,
'activity_date':'datetime64[ns]'})
# 1. 只保留 login 行
login = traffic[traffic['activity'] == 'login']

# 2. 每个用户首次登录日期
first_login = login.groupby('user_id', as_index=False)['activity_date'].min()
first_login.rename(columns={'activity_date': 'first_login'}, inplace=True)

# 3. 过滤最近 90 天内首次登录的用户
start_date = pd.to_datetime('2019-06-30') - pd.Timedelta(days=90)

filtered = first_login[
    (first_login['first_login'] >= start_date) &
    (first_login['first_login'] <= '2019-06-30')
]

# 4. 按首次登录日期计数
result = filtered.groupby('first_login', as_index=False)['user_id'].count()

# 5. 改名成题目要求格式
result.rename(columns={
    'first_login': 'login_date',
    'user_id': 'user_count'
}, inplace=True)

result


,login_date,user_count
0,2019-05-01,1
1,2019-06-21,2
